# Generate repayment_behavior.csv

This notebook reads `data/loan_outcome.csv` and `data/loan_details.csv` and creates a month-level repayment schedule for each loan following the Stage‑3 repayment behavior rules:
- Non-defaulted loans: mostly on-time payments (DPD=0), occasional small late days.
- Defaulted loans: worsening DPD pattern in months leading up to default, then missed payments after default.

Output: `data/repayment_behavior.csv` with columns: repayment_id, loan_id, customer_id, month_number, due_date, payment_date, amount_due, amount_paid, dpd, payment_status, bounce_flag.

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [13]:
# Read inputs
loan_path = Path('../data/loan_outcome.csv')
loan_details_path = Path('../data/loan_details.csv')
if not loan_path.exists() or not loan_details_path.exists():
    raise FileNotFoundError('Ensure data/loan_outcome.csv and data/loan_details.csv exist (run previous notebooks)')
lo = pd.read_csv(loan_path)
ld = pd.read_csv(loan_details_path).set_index('loan_id')
# join emi_amount and tenure if not present in outcome
if 'emi_amount' not in lo.columns or 'tenure_months' not in lo.columns:
    lo = lo.merge(ld[['emi_amount','tenure_months','origination_date']], on='loan_id', how='left')
lo['origination_date'] = pd.to_datetime(lo['origination_date'])
print('Loaded', len(lo), 'loans for repayment generation')

Loaded 12000 loans for repayment generation


In [14]:
# Helper to generate monthly rows for one loan
def generate_repayment_rows(row, start_id):
    rows = []
    loan_id = row['loan_id']
    cust = row.get('customer_id', None)
    tenure = int(row.get('tenure_months', 12))
    emi = float(row.get('emi_amount', 0))
    orig = pd.to_datetime(row.get('origination_date'))
    default_flag = int(row.get('default_flag', 0)) if not pd.isna(row.get('default_flag')) else 0
    default_month = int(row['default_month']) if (default_flag==1 and not pd.isna(row.get('default_month'))) else None
    rid = start_id
    for m in range(1, tenure+1):
        due_date = (orig + pd.DateOffset(months=m-1)).date()
        # Determine dpd and payment behavior
        if default_flag==0:
            # mostly on time, small chance of late up to 15 days
            if random.random() < 0.95:
                dpd = 0
                amount_paid = emi
                payment_date = due_date
                status = 'On Time'
            else:
                dpd = int(np.random.choice([1,3,5,10,15], p=[0.4,0.25,0.2,0.1,0.05]))
                # if slightly late, usually full payment; if >10 days, partial possible
                amount_paid = emi if dpd <=10 else round(emi * np.random.uniform(0.3,0.9),2)
                payment_date = (pd.to_datetime(due_date) + pd.Timedelta(days=dpd)).date()
                status = 'Late' if amount_paid>0 else 'Missed'
        else:
            # defaulted loan: worsening pattern before default, then missed payments after default
            if default_month is not None and m < default_month - 2:
                # early months: mostly on time or small lateness
                if random.random() < 0.85:
                    dpd = 0
                    amount_paid = emi
                    payment_date = due_date
                    status = 'On Time'
                else:
                    dpd = int(np.random.choice([1,3,5], p=[0.6,0.3,0.1]))
                    amount_paid = emi
                    payment_date = (pd.to_datetime(due_date) + pd.Timedelta(days=dpd)).date()
                    status = 'Late'
            elif default_month is not None and m >= default_month - 2 and m < default_month:
                # deterioration window: larger late days and partial payments
                dpd = int(np.random.choice([5,10,15,30], p=[0.2,0.3,0.3,0.2]))
                # partial payment likely
                amount_paid = round(emi * np.random.uniform(0.0,0.7),2)
                payment_date = (pd.to_datetime(due_date) + pd.Timedelta(days=dpd)).date() if amount_paid>0 else None
                status = 'Late' if amount_paid>0 else 'Missed'
            elif default_month is not None and m >= default_month:
                # after default: missed payments
                dpd = min(180, int((m - default_month + 1) * 30))
                amount_paid = 0.0
                payment_date = None
                status = 'Missed'
            else:
                # fallback
                dpd = 0
                amount_paid = emi
                payment_date = due_date
                status = 'On Time'
        # bounce flag: small chance on missed/late payments
        bounce = 1 if dpd>0 and random.random() < 0.05 else 0
        rows.append({
            'repayment_id': f'REP{str(rid).zfill(8)}',
            'loan_id': loan_id,
            'customer_id': cust,
            'month_number': m,
            'due_date': due_date.isoformat(),
            'payment_date': payment_date.isoformat() if payment_date is not None else None,
            'amount_due': round(emi,2),
            'amount_paid': round(amount_paid,2) if amount_paid is not None else 0.0,
            'dpd': int(dpd) if dpd is not None else None,
            'payment_status': status,
            'bounce_flag': bounce
        })
        rid += 1
    return rows, rid

# Generate for all loans (this can create ~200k+ rows depending on tenures)
all_rows = []
next_id = 1
for _, r in lo.iterrows():
    try:
        rows, next_id = generate_repayment_rows(r, next_id)
        all_rows.extend(rows)
    except Exception as e:
        print('Error generating for loan', r.get('loan_id'), e)

rep_df = pd.DataFrame(all_rows)
rep_df['dpd'] = rep_df['dpd'].clip(upper=180)
print('Generated repayment rows:', len(rep_df))
Path('../data').mkdir(parents=True, exist_ok=True)
out_path = Path('../data/repayment_behavior.csv')
rep_df.to_csv(out_path, index=False)
print(f'Wrote repayment behavior to: {out_path}')

Generated repayment rows: 217383
Wrote repayment behavior to: ..\data\repayment_behavior.csv


In [15]:
# Quick validations
print('DPD summary:')
print(rep_df['dpd'].describe())
print('Payment status counts:')
print(rep_df['payment_status'].value_counts(dropna=False).head())
print('Sample rows:')
print(rep_df.head().to_string(index=False))

DPD summary:
count    217383.000000
mean         24.249854
std          58.359635
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         180.000000
Name: dpd, dtype: float64
Payment status counts:
payment_status
On Time    169434
Missed      34035
Late        13914
Name: count, dtype: int64
Sample rows:
repayment_id     loan_id customer_id  month_number   due_date payment_date  amount_due  amount_paid  dpd payment_status  bounce_flag
 REP00000001 LOAN0000001  CUST006553             1 2023-07-02   2023-07-02     9927.66      9927.66    0        On Time            0
 REP00000002 LOAN0000001  CUST006553             2 2023-08-02   2023-08-02     9927.66      9927.66    0        On Time            0
 REP00000003 LOAN0000001  CUST006553             3 2023-09-02   2023-09-02     9927.66      9927.66    0        On Time            0
 REP00000004 LOAN0000002  CUST000807             1 2023-10-31   2023-10-31     8363.06      8363.06    0        On